In [1]:
import openai
from qdrant_client import QdrantClient

embedding function

In [2]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

Retrieval function

In [3]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [4]:
def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [5]:
retrieved_context = retrieve_data("Do you have a USB connectable fan for hot summers.", k=10)

In [6]:
retrieved_context

{'retrieved_context_ids': ['B0C6JRWWFQ',
  'B09ZVFHMJN',
  'B0B2WB18Q4',
  'B0BWHHJQQ9',
  'B0B7H3D2R9',
  'B0CC17CH1N',
  'B09SZ7222N',
  'B0CCJ9PTB1',
  'B0B931F128',
  'B0BGLJC82D'],
 'retrieved_context': ['Hotsales Portable Neck Fan, Hands Free Bladeless Neck Fans Portable Rechargeable, 360° Cooling Neck Fan, 4 Speeds, Quiet, USB Powered, 5200 mAh Battery Operated Wearable Personal Fan for Women, Men ',
  'USB Outlet Extender Surge Protector with Rotating Plug, 6 AC Multi Plug Outlet and 3 USB Ports , 1680 Joules, 3-Sided Swivel Power Strip with Spaced Outlet Splitter for Home, Office, Travel Multi USB Wall charger: 6 AC outlets splitter with 3 smart USB port power for cell phones, laptops, cameras and many other devices. With built-in surge protector (1680J), it can ensure your precious devices are under protection. Smart USB Charging Ports: 3 USB charging ports(2USB A+1 USB C) total 3A. Multi plug outlet can auto-detect your devices and seek to maximize the fastest possible charg

Format retrieved context function

In [7]:
def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

In [8]:
preprocessed_context = process_context(retrieved_context)

In [9]:
print(preprocessed_context)

- ID: B0C6JRWWFQ, rating: 4.2, description: Hotsales Portable Neck Fan, Hands Free Bladeless Neck Fans Portable Rechargeable, 360° Cooling Neck Fan, 4 Speeds, Quiet, USB Powered, 5200 mAh Battery Operated Wearable Personal Fan for Women, Men 
- ID: B09ZVFHMJN, rating: 4.6, description: USB Outlet Extender Surge Protector with Rotating Plug, 6 AC Multi Plug Outlet and 3 USB Ports , 1680 Joules, 3-Sided Swivel Power Strip with Spaced Outlet Splitter for Home, Office, Travel Multi USB Wall charger: 6 AC outlets splitter with 3 smart USB port power for cell phones, laptops, cameras and many other devices. With built-in surge protector (1680J), it can ensure your precious devices are under protection. Smart USB Charging Ports: 3 USB charging ports(2USB A+1 USB C) total 3A. Multi plug outlet can auto-detect your devices and seek to maximize the fastest possible charge speed, usb-a up to 2.4A per port, usb-c up to 3A. 180° Rotating AC Plug: 90° left or right rotating AC plug. Whether your wal

Create prompt template function

In [10]:
def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt

In [11]:
prompt = build_prompt(preprocessed_context, "Do you have a USB connectable fan for hot summers?")

In [12]:
print(prompt)


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
- ID: B0C6JRWWFQ, rating: 4.2, description: Hotsales Portable Neck Fan, Hands Free Bladeless Neck Fans Portable Rechargeable, 360° Cooling Neck Fan, 4 Speeds, Quiet, USB Powered, 5200 mAh Battery Operated Wearable Personal Fan for Women, Men 
- ID: B09ZVFHMJN, rating: 4.6, description: USB Outlet Extender Surge Protector with Rotating Plug, 6 AC Multi Plug Outlet and 3 USB Ports , 1680 Joules, 3-Sided Swivel Power Strip with Spaced Outlet Splitter for Home, Office, Travel Multi USB Wall charger: 6 AC outlets splitter with 3 smart USB port power for cell phones, laptops, cameras and many other devices. With built-in surge protector (1680J), it can ensure your precious d

Generate answer function

In [13]:
def generate_answer(prompt):

    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning_effort="none"
    )

    return response.choices[0].message.content

In [14]:
print(generate_answer(prompt))

Yes. We have a portable neck fan (Hotsales Portable Neck Fan, bladeless, 360° cooling, 4 speeds, quiet). It’s hands-free, rechargeable (5200 mAh), and USB powered for hot summer cooling.


Combined RAG pipeline

In [15]:
def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    return answer

In [16]:
print(rag_pipeline("Do you have a USB connectable fan for hot summers?"))

Yes. We have a Hotsales Portable Neck Fan (B0C6JRWWFQ) that’s a hands-free, rechargeable 360° cooling neck fan with 4 speeds. It’s USB powered and uses a 5200 mAh battery, designed for hot summer days.


In [17]:
print(rag_pipeline("Could you suggest me some earphones? I am only interested in the ones that have above 4 rating.", 10))

Sure! Here are earphones/headphones from the available products with a rating above 4:

1) Jesebang Wireless Earbud (Bluetooth Headphones 5.3, 40H, ENC Mic, USB-C LED display, IP7 waterproof, sport earhooks) — Rating 4.9 (Black)

2) Cowyawn Toddler Headphones (wired, ultra-light, rotatable fit, safe volume control 85/94dB, built-in microphone, 3.5mm jack) — Rating 4.5 (Navy/Teal)

3) Earbay Bluetooth Headset with Microphone (ENC noise-cancelling mic, USB dongle, mute button, up to 26hrs talk time, multipoint) — Rating 4.0 (Rose Gold)

4) LUDOS Turbo Wired Earbuds (in-ear, ergonomic design, microphone, memory foam, durable cable) — Rating 4.4

If you tell me whether you prefer wired or wireless (and for what device—iPhone Lightning, USB-C, or PC), I can narrow it down to the best match.


In [18]:
print(rag_pipeline("Could you suggest me some earphones? I am only interested in the ones that have bellow 4 rating.", 10))

Here are earphones/headphones from the available products with a rating below 4.0 (strictly less than 4):

1) LUDOS Turbo Wired Earbuds with Microphone (B07VSWK14G) — Rating: 4.4 (This one is above 4, so I’m skipping it)

2) Cowyawn Toddler Kids Wired Headphones (B0C2VVFQLD) — Rating: 4.5 (Above 4, skipping)

3) Bluetooth Headset with Microphone, USB Dongle & Mute Button (B0C6KM4H9M) — Rating: 4.0 (Not below 4, so I’m skipping it)

If you meant “4.0 or lower” instead of “below 4”, then the only one that matches is:
- Bluetooth Headset with Microphone, USB Dongle & Mute Button (B0C6KM4H9M) — Rating: 4.0

Tell me which you meant (below 4 vs 4.0 or lower) and what device you’ll use (phone/iPhone, PC/Zoom, kids, etc.).
